# Privacy Metric Example

The **Privacy** metric scores how suitable a PII/PHI detector is for your domain, and `PrivacyRanker` orders several detectors so you can pick one. The score combines five factors:

```
Score = DetectionScore × Coverage × DomainFit × RegulatoryFit × Penalty_FN
```

plus a complementary **risk index** (`R_final`) that flags critical blind spots. This notebook runs with **no extra dependencies** — it uses a tiny regex detector so you can see the full input → output shape. Swap in `PresidioDetector` / `HuggingFacePIIDetector` once you install the corresponding extra.

## Installation

In [ ]:
# Metric core needs no extras. For real backends use one of:
#   !pip install "gaussia[privacy-presidio]" matplotlib -q
#   !pip install "gaussia[privacy-huggingface]" matplotlib -q
!pip install "gaussia" matplotlib -q

## Setup

Three pieces: a **domain config** (which classes matter + their weights), one or more **detectors**, and a **retriever** that yields labelled `PrivacyBatch`es (`query` = text shown to the detector, `spans` = ground truth).

In [ ]:
import re

from gaussia.core.detector import PIIDetector
from gaussia.core.retriever import Retriever
from gaussia.metrics.privacy import Privacy, PrivacyRanker
from gaussia.schemas.common import Dataset
from gaussia.schemas.privacy import PrivacyBatch, PrivacyDomainConfig, Span

# The evaluation domain. Both weight maps must sum to 1.0.
#   criticality_weights -> how much each class counts toward the DetectionScore
#   fn_severity_weights -> how costly a MISS of each class is (drives the risk index)
DOMAIN = PrivacyDomainConfig(
    classes=frozenset({"email_address", "phone_number"}),
    criticality_weights={"email_address": 0.6, "phone_number": 0.4},
    fn_severity_weights={"email_address": 0.7, "phone_number": 0.3},
    iou_threshold=0.5,
    regulatory_framework="GDPR",
)

_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_PHONE = re.compile(r"\d{3}-\d{4}")


class RegexDetector(PIIDetector):
    """A trivial detector so the notebook runs without optional backends."""

    @property
    def supported_classes(self) -> frozenset[str]:
        return frozenset({"email_address", "phone_number"})

    def predict(self, text: str) -> list[Span]:
        spans = [Span(label="email_address", start=m.start(), end=m.end(), text=m.group(), score=0.99) for m in _EMAIL.finditer(text)]
        spans += [Span(label="phone_number", start=m.start(), end=m.end(), text=m.group(), score=0.90) for m in _PHONE.finditer(text)]
        return spans


class EmailOnlyDetector(PIIDetector):
    """A weaker detector: only finds emails -> lower coverage and a phone blind spot."""

    @property
    def supported_classes(self) -> frozenset[str]:
        return frozenset({"email_address"})

    def predict(self, text: str) -> list[Span]:
        return [Span(label="email_address", start=m.start(), end=m.end(), text=m.group(), score=0.99) for m in _EMAIL.finditer(text)]


class InMemoryRetriever(Retriever):
    """Yields one labelled conversation. Replace with your own Retriever subclass."""

    def load_dataset(self) -> list[Dataset]:
        text = "Reach me at john@example.com or 555-1234."
        turn = PrivacyBatch(
            qa_id="turn-1",
            query=text,
            assistant="",
            ground_truth_assistant="",
            spans=[
                Span(label="email_address", start=12, end=28, text="john@example.com"),
                Span(label="phone_number", start=32, end=40, text="555-1234"),
            ],
        )
        return [Dataset(session_id="demo", assistant_id="bot", context="", conversation=[turn])]

## Evaluate a single detector

`Privacy.run` returns one `PrivacyMetric` per dataset. To use a real backend, replace the detector with e.g. `PresidioDetector(name="presidio", domain_fit=0.9, regulatory_fit=0.8)` after installing `gaussia[privacy-presidio]`.

In [ ]:
detector = RegexDetector(name="regex-baseline", domain_fit=0.9, regulatory_fit=0.8)
metric = Privacy.run(InMemoryRetriever, detector=detector, domain_config=DOMAIN)[0]

print(f"detector        : {metric.name}")
print(f"score (0-100)   : {metric.score_100:.2f}  -> {metric.interpretation}")
print(f"detection_score : {metric.detection_score:.3f}")
print(f"coverage        : {metric.coverage:.3f}")
print(f"penalty_fn      : {metric.penalty_fn:.3f}")
print(f"risk (r_final)  : {metric.r_final:.3f}  (weakest class: {metric.r1_weakest_class})")
print("\nper-class breakdown:")
for label, cm in metric.class_metrics.items():
    print(f"  {label:<14} f2={cm.f2:.2f}  tp={cm.tp} fp={cm.fp} fn={cm.fn}")

## Rank several detectors

`PrivacyRanker.run` evaluates every detector against the same corpus and returns a `PrivacyRanking` ordered by `score_100` (failures sorted to the tail; if a detector raises it is recorded with `success=False` and the ranking still completes).

In [ ]:
good = RegexDetector(name="regex-baseline", domain_fit=0.9, regulatory_fit=0.8)
weak = EmailOnlyDetector(name="email-only", domain_fit=0.5, regulatory_fit=0.5)

ranking = PrivacyRanker.run(InMemoryRetriever, detectors=[good, weak], domain_config=DOMAIN)[0]

print(f"winner: {ranking.winning_detector}\n")
for rank, result in enumerate(ranking.results, start=1):
    print(f"  {rank}. {result.name:<16} score_100={result.score_100:6.2f}  success={result.success}")

## Visualize the ranking

In [ ]:
import matplotlib.pyplot as plt

names = [r.name for r in ranking.results]
scores = [r.score_100 for r in ranking.results]

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(names[::-1], scores[::-1], color="#4C72B0")
ax.set_xlabel("score_100")
ax.set_xlim(0, 100)
ax.set_title("Detector ranking (higher is better)")
for i, s in enumerate(scores[::-1]):
    ax.text(s + 1, i, f"{s:.1f}", va="center")
plt.tight_layout()
plt.show()

## Understanding the results

- **`score_100`** is the headline number; **`interpretation`** maps it to a qualitative band (`"Not suitable"` … `"Recommended with strong local evidence"`).
- The score is the **product** of five factors, so a single weak factor (e.g. a detector that doesn't *support* a class → low `coverage`, or misses a critical class → low `penalty_fn`) drags the whole score down. That is intentional.
- **`r_final`** is the risk index: it stays high when a detector has a critical blind spot even if its average score looks acceptable — inspect `r1_weakest_class` to see which class.
- **`load_time` / `inference_latency`** are reported for diagnostics only and never enter the score.
- To plug in your own model, subclass `PIIDetector` (implement `predict` and `supported_classes`) — no library change needed, and it slots straight into `Privacy` / `PrivacyRanker`.